# CR-2 G2c 確かめる段（Claude バッチ API・CPU）

Git に固定した段3b の記録のとおりに、**確かめる段**のリクエストを送ります。

- **送る前に（G2c の前の受け入れ）**：(1) 段3b のコミットがこのコードの祖先であること、(2) 探す段の生の結果から作り直した段3b の記録が、**Git に固定したもの**と **Drive の記録**の両方と一致することを確かめます。どれか 1 つでも違えば、1 件も送りません。
- 確かめる段の引き直しは、すべて新しく頼みます（探す段の応答は使いません）。各バッチの記録に段3b のハッシュを書きます。
- お金の決まりは G2s と同じ（**$54** の線、送る前の確かめ、途中で止める条件）。$60 を超えない保証ではありません。
- 途中で切れても、セル1 → セル2 → セル3 で続きから回ります。ランタイムは 1 つだけ。
- 問題文・応答は表示しません。

**使い方**：CPU のランタイム。シークレットは `ANTHROPIC_API_KEY` と `GITHUB_TOKEN`。セル1 → セル2 → セル3。数時間かかることがあります。終わったら Claude に教えてね。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。
- **エラーのとき**：通知の本文にエラーの文面（トレースバック）が入ります。全文は Drive の `DECISION_POINTS/_errors/` にも保存されるので、Claude が直接読めます。


In [ ]:
# Cell 1 — Drive, exact head, ancestors, (pinned SDK and key, never printed), notifications.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time, hashlib, datetime as dt

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。エラーのときはエラーの文面も送ります（鍵・トークンらしい文字は伏せ字。全文は Drive の DECISION_POINTS/_errors）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'CR-2 G2c 確かめる段（バッチAPI・CPU）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'Pork - CR-2 G2c confirmation', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
import os, re as _re, traceback as _tb, datetime as _edt
ERR_SLUG = 'CR2_G2C'   # Drive のエラーファイルの名前の頭
_SECRET_RE = _re.compile(r"sk-ant-[A-Za-z0-9_\-]+|github_pat_[A-Za-z0-9_]+|gh[pousr]_[A-Za-z0-9]{8,}|hf_[A-Za-z0-9]{8,}|x-access-token:[^@\s]+@")
def _scrub(s):
    s = _SECRET_RE.sub('[伏せ字]', s)
    return '\n'.join(l if len(l) <= 300 else l[:300] + ' …（長いので省略）' for l in s.splitlines())
def _error_text(result):
    e = result.error_in_exec if result.error_in_exec is not None else result.error_before_exec
    tb = ''.join(_tb.format_exception(type(e), e, e.__traceback__))
    first = (getattr(getattr(result, 'info', None), 'raw_cell', '') or '').strip().splitlines()[:1]
    return _scrub('セル: ' + (first[0][:100] if first else '?') + '\n' + tb)
def _save_error(text):
    base = '/content/drive/MyDrive'
    if not os.path.isdir(base):
        return None
    d = os.path.join(base, 'DECISION_POINTS', '_errors')
    os.makedirs(d, exist_ok=True)
    stem = ERR_SLUG + '_' + _edt.datetime.now(_edt.timezone(_edt.timedelta(hours=9))).strftime('%Y%m%d_%H%M%S')
    name, n = stem + '.txt', 1
    while os.path.exists(os.path.join(d, name)):
        n += 1
        name = stem + '_' + str(n) + '.txt'
    with open(os.path.join(d, name), 'w', encoding='utf-8') as fh:
        fh.write('[' + NB_LABEL + ']\n' + text)
    return 'DECISION_POINTS/_errors/' + name
def _short(text, limit=3300):
    b = text.encode('utf-8')
    if len(b) <= limit:
        return text
    return '…（前のほうは省略。全文は Drive）\n' + b[-limit:].decode('utf-8', 'ignore')
def _on_error(result):
    if result.error_in_exec is None and result.error_before_exec is None:
        return
    try:
        text = _error_text(result)
    except Exception as e2:
        text = 'エラーの文面を作れませんでした：' + type(e2).__name__
    try:
        where = _save_error(text)
    except Exception as e3:
        where = None
        text += '\n（Drive に保存できませんでした：' + type(e3).__name__ + '）'
    print('エラーの全文を通知に送ります' + ('（Drive：' + where + '）' if where else ''))
    notify('止まりました：エラーが出ています。' + ('全文は Drive の ' + where if where else 'Drive には保存できませんでした')
           + '\n\n' + _short(text), tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

drive.mount("/content/drive")
EXPECTED_HEAD = "c2659a50138f5983f2528f42d711e257c94bf579"
ANCESTORS = ["311e3008211ebaca0735abccf3501e6fcb8da3df", "da200a4805bdd727f4bb4b4094a08aa070c31a47", "da200a4805bdd727f4bb4b4094a08aa070c31a47", "c2659a50138f5983f2528f42d711e257c94bf579"]
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")

def get_secret(name):
    try:
        v = userdata.get(name)
    except Exception:
        v = None
    return str(v).strip() if v and str(v).strip() else None

token = get_secret("GITHUB_TOKEN")
assert token, "Colab Secrets に GITHUB_TOKEN を登録してください。"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
os.chdir('/content')   # セル1 をやり直しても、消すフォルダの中にいないように（2026-10-14）
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--branch", "decision-points", "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
for anc in ["7949d74"] + ANCESTORS:
    assert subprocess.run(["git", "-C", str(REPO), "merge-base", "--is-ancestor", anc, head]).returncode == 0, anc + " が祖先ではありません"
token = None
env.pop("GITHUB_TOKEN", None)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
from pork import cr2 as C
from pork import cr2_search as S
from pork import cr2_run as RN
from pork import cr2_g2 as G
from pork import cr2_g3 as G3
from pork import cr2_stage3b as T3
from pork import cr1_run as R1
from pork import alt_prelook as P
import numpy
RN.check_prompt()
D = Path("/content/drive/MyDrive/DECISION_POINTS/cr2")
D.mkdir(parents=True, exist_ok=True)
JST = dt.timezone(dt.timedelta(hours=9))

def git_blob(path):
    return subprocess.check_output(["git", "hash-object", path], text=True).strip()

def write_json(path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    R1._atomic_write(path, obj)
    return hashlib.sha256(path.read_bytes()).hexdigest()

def load_chase_verified():
    """Plan 1. (H4, I3): pinned revision, canonical hash from the Hugging Face API vs the downloaded bytes."""
    from huggingface_hub import HfApi, hf_hub_download
    info = HfApi().dataset_info(C.DATASET_ID, revision=C.DATASET_REVISION, files_metadata=True, token=False)
    sib = next((s for s in info.siblings if s.rfilename == C.DATASET_FILE), None)
    meta = None
    if sib is not None:
        lfs = sib.lfs
        if lfs is not None and not isinstance(lfs, dict):
            lfs = {"sha256": getattr(lfs, "sha256", None), "size": getattr(lfs, "size", None)}
        meta = {"blob_id": sib.blob_id, "lfs": lfs}
    p = hf_hub_download(C.DATASET_ID, C.DATASET_FILE, repo_type="dataset", revision=C.DATASET_REVISION, token=False)
    data = Path(p).read_bytes()
    prov = C.check_provenance(data, meta, info.sha)
    rp = hf_hub_download(C.DATASET_ID, "README.md", repo_type="dataset", revision=C.DATASET_REVISION, token=False)
    readme = Path(rp).read_text(encoding="utf-8")
    rows = C.parse_rows(data)
    del data
    return rows, prov, {"readme_sha256": C.sha256_text(readme), "license": C.readme_license(readme)}

def rebuild_stage2(file_sha=None, cand_sha=None, excl_sha=None):
    rows, prov, readme = load_chase_verified()
    if file_sha is not None:
        assert prov["file_sha256"] == file_sha, "データのファイルが段2 の固定と違います。止めます（1 件も送りません）。"
    pre = C.reproduce_prelook3(rows, P.load_math())
    cand = C.build_candidates(rows, pre)
    if cand_sha is not None:
        assert cand["candidates_sha256"] == cand_sha, "候補の一覧が段2 の固定と違います。止めます。"
    if excl_sha is not None:
        assert cand["excluded_sha256"] == excl_sha, "除外した行が段2 の固定と違います。止めます。"
    by_id = {int(r["ID"]): r for r in rows}
    cands = [{**c, "question": C.problem_text(by_id[c["id"]]), "gold": str(C.gold(by_id[c["id"]]))}
             for c in cand["candidates"]]                                     # PRIVATE, memory only
    del rows, by_id
    return cand, cands, prov, readme, pre

r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic==" + RN.SDK_VERSION], capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-2000:]
import anthropic
assert anthropic.__version__ == RN.SDK_VERSION, anthropic.__version__
_key = get_secret("ANTHROPIC_API_KEY")
assert _key, "Colab のシークレットに ANTHROPIC_API_KEY を登録してね。"
client = anthropic.Anthropic(api_key=_key)                       # reads, count_tokens: default retries
create_client = anthropic.Anthropic(api_key=_key, max_retries=0)  # sends: never retried by the SDK
del _key
TRANSIENT = (anthropic.RateLimitError, anthropic.APIConnectionError, anthropic.InternalServerError, anthropic.APITimeoutError)
print("anthropic", anthropic.__version__)

print("study:", C.STUDY, "| exact head:", head, "| numpy", numpy.__version__, "| python", sys.version.split()[0])


In [ ]:
# Cell 2 — G2c の前の受け入れ (three-way 段3b check).
OWNER_CONFIRMS_OLD_RUNTIME_STOPPED = ""
G1_HEAD = "311e3008211ebaca0735abccf3501e6fcb8da3df"
G2S_HEAD = "da200a4805bdd727f4bb4b4094a08aa070c31a47"
CANDIDATES_SHA = "c4bffbfa9632ed51a1222fae0bc65b256b3db0ae7e1b0d12e8bf70ac649f2339"
FILE_SHA = "731f2db9a590ac6c49ab7529d08dee0fcb09cc832a6462248e6939919319222c"
FINAL_ITEMS_SHA = "9ffd511cff0024e5106e50d18b5559f179485d10d528c973a2f458648ab8fee9"
STAGE3B_SHA = "1a70cc32b83dab56257911fa3c1ad985b30d6dbc64448907e2026ecd82b02343"
STAGE3B_COMMIT = "c2659a50138f5983f2528f42d711e257c94bf579"

assert subprocess.run(["git", "diff", "--quiet", G1_HEAD, head, "--", "pork", "scripts", "tests"]).returncode == 0, \
    "G1 のあとでコードが変わっています。止めます（1 件も送りません）。"
LOCK3 = json.loads((REPO / "notes/cr2_lock_stage3.json").read_text())                  # the committed 段3 lock
assert LOCK3["code_blobs"] == {"pork/cr2_search.py": git_blob("pork/cr2_search.py"), "pork/cr2.py": git_blob("pork/cr2.py")}, "定義 17・18 のコードが段3 の固定と違います。止めます。"
assert LOCK3["numpy"] == numpy.__version__, "numpy の版が段3 の固定と違います。止めます（Claude に教えてね）。"
assert LOCK3["sdk"] == RN.SDK_VERSION and LOCK3["price_usd_per_mtok"] == {"in": RN.PRICE_IN, "out": RN.PRICE_OUT, "batch_factor": 0.5}
assert LOCK3["budget"]["B_stop"] == C.B_STOP and LOCK3["budget"]["B_cap"] == C.B_CAP and LOCK3["budget"]["worst_output_tokens"] == C.MAX_TOKENS
assert LOCK3["final_items_sha256"] == FINAL_ITEMS_SHA and LOCK3["file_sha256"] == FILE_SHA
assert LOCK3["rules"] == {"grid": S.GRID, "stop_width": S.STOP_WIDTH, "K_S": S.K_S, "min_returned": S.MIN_RETURNED,
                          "controls": S.N_CONTROLS, "K": C.K, "K_A": C.K_A, "SEED_CR2_WORD": C.SEED_CR2_WORD,
                          "control_key": "default_rng([SEED_CR2_WORD, ID])"}, "探す段・確かめる段の決まりが段3 の固定と違います。止めます。"

cand, cands, prov, readme, pre = rebuild_stage2(FILE_SHA, CANDIDATES_SHA)
P_ = RN.paths()
ck = json.loads((Path("/content/drive/MyDrive") / P_["checkpoint"]).read_text())
assert ck["header"] == RN.header(G1_HEAD, cand["candidates_sha256"], CANDIDATES_SHA, RN.SDK_VERSION, C.DATASET_REVISION, FILE_SHA, FILE_SHA)
RN.validate(ck, cands)
g0 = json.loads((D / "G0/CR2_G0.json").read_text())
assert g0["pass"] and g0["code_head"] == G1_HEAD
g1 = RN.summarize_g1(ck, cands, g0["spent_usd"])
assert g1["verdict"] == "GO" and g1["final_items_sha256"] == FINAL_ITEMS_SHA, "G1 の結果が段3 の固定と違います。止めます。"
questions = {C.cid(c["id"]): c["question"] for c in cands}
ITEMS = RN.manifest_items(g1, ck, questions)
PRIOR = g0["spent_usd"] + g1["g1_spent_usd"]
PER_G1B = g1["G1b"]["estimate"]["per_request_usd_base"]      # early_stop applies the x1.2
del questions, cands

OUTS = D / "G2S"
states = []
for r in range(T3.MAX_ROUNDS):
    p = OUTS / f"CR2_G2S_round{r}_state.json"
    if not p.exists():
        break
    states.append(json.loads(p.read_text()))
    assert states[-1]["header"]["code_head"] == G2S_HEAD
ROUND_RESULTS, MANIFESTS = G.search_round_results(ITEMS, states, OUTS)
rebuilt = T3.stage3b_record(ITEMS, T3.plan_search(ITEMS, ROUND_RESULTS), MANIFESTS, FINAL_ITEMS_SHA)
committed = (REPO / "notes/cr2_lock_stage3b.json").read_text()
at_commit = subprocess.check_output(["git", "show", STAGE3B_COMMIT + ":notes/cr2_lock_stage3b.json"], text=True)
assert at_commit == committed, "Git の段3b の記録が、段3b のコミットのものと違います。止めます。"
drive_rec = (D / "STAGE3B/CR2_stage3b_record.json").read_text()
h3b = T3.verify_stage3b(rebuilt, committed, drive_rec)
assert h3b == STAGE3B_SHA, "段3b のハッシュがノートブックの値と違います。止めます（1 件も送りません）。"
REC = json.loads(committed)
CONF = T3.confirmation_items(REC)
G2S_SPENT = G.stage_spent(states)
PER = G.mean_actual_per_request(states) or PER_G1B
del states, ROUND_RESULTS, rebuilt, drive_rec
OUT = D / "G2C"
OUT.mkdir(parents=True, exist_ok=True)   # 初めての G2c ではフォルダがまだないため（2026-10-14 v3）
R1.drive_smoke(OUT)
REQ = G.Requests(ITEMS, CONF, "c", REC["confirmation_manifest_sha256"])

from pork import cr1_g2 as G1x
API = G1x.AnthropicBatchAPI(client, create_client)
import uuid
_rid = Path("/content/g2_runtime_id")
if not _rid.exists():
    _rid.write_text(uuid.uuid4().hex[:12])
RUNTIME = _rid.read_text().strip()
print("このランタイムの番号:", RUNTIME)

def run_list(ST, REQ, save, fence, label):
    last = None
    while True:
        try:
            fence()
            G.claim_lease(ST, RUNTIME, dt.datetime.now(JST), save)
            G.collect(ST, REQ, API, OUT, save)
            if ST["submitting"]:
                G.resolve_submitting(ST, API, save, dt.datetime.now(JST))
            G.submit_ready(ST, REQ, API, save, dt.datetime.now(JST), API.count_tokens, fence_check=fence)
        except TRANSIENT as e:
            print("一時的なエラー:", type(e).__name__, "→ 1 分後にもう一度")
            time.sleep(60)
            continue
        save()
        if ST["stopped"] or G.done(ST, REQ):
            break
        t = G.tallies(ST, REQ)
        line = (f"{label} | 成功 {len(t['succeeded'])}/{len(REQ)} | 流れている {len(t['inflight'])} | 残りのかたまり {len(t['chunks_left'])}"
                f" | 出し直し待ち {len(t['retry'])} | 確定した額 ${G.spent_usd(ST):.2f}")
        if line != last:
            print(dt.datetime.now(JST).strftime("%H:%M"), line)
            last = line
        time.sleep(60)
    G.verify_collected(ST, REQ, OUT, api=API, save=save)

def open_list(path, HDR, REQ):
    if path.exists():
        saved = json.loads(path.read_text())
        if saved["header"]["code_head"] != head:
            raise G.G2Stop("the state was written by another code commit; ask Claude before resuming")
        ST = G.open_state(saved, HDR)
    else:
        ST = G.open_state(None, HDR)
    save = lambda: G.save_state(path, ST)
    fence = lambda: G.fence(ST, lambda: json.loads(path.read_text()))
    G.claim_lease(ST, RUNTIME, dt.datetime.now(JST), save, owner_confirms_stopped=OWNER_CONFIRMS_OLD_RUNTIME_STOPPED or None)
    G.verify_collected(ST, REQ, OUT, api=API, save=save)
    G.resolve_submitting(ST, API, save, dt.datetime.now(JST))
    save()
    return ST, save, fence

HDR = G.g2_header(REC["confirmation_manifest_sha256"], head, anthropic.__version__, PRIOR + G2S_SPENT, PER, "c", None, 0, h3b)
ST, save, fence = open_list(OUT / "CR2_G2C_state.json", HDR, REQ)
t = G.tallies(ST, REQ)
print("G2c の前の受け入れ: OK（段3b の三つが一致） | リクエスト", len(REQ), "| 成功", len(t["succeeded"]),
      "| 確定した額 $%.2f" % G.spent_usd(ST))

In [ ]:
# Cell 3 — the confirmation stage.
run_list(ST, REQ, save, fence, "確かめる段")
S_ = G.summary(ST, REQ)
sha = write_json(OUT / "CR2_G2C_summary.json", S_)
print("終わり:", "完了" if S_["done"] else S_["stopped"], "| 成功", S_["succeeded"], "| 解決しない", S_["unresolved"],
      "| 合計 $%.2f" % S_["spent_total_usd"], "| 各バッチに段3b のハッシュ:", S_["stage3b_sha256_in_every_batch"], "| summary sha256", sha)
msg = '終わりました：CR-2 の確かめる段の全バッチを集めました' if S_["done"] else '止まりました：計画の決まりで確かめる段を止めました（Claude に見せてね）'
notify(msg, tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()